In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
from xgboost import XGBClassifier

In [ ]:
from sklearn.ensemble import VotingClassifier

In [ ]:
from sklearn.metrics import accuracy_score , classification_report

In [ ]:
from tensorflow.keras.models import Sequential

In [ ]:
from tensorflow.keras.layers import Dense , LSTM , Conv1D , Flatten

In [ ]:
!pip install tensorflow xgboost scikit-learn pandas numpy


In [2]:
from google.colab import files

uploaded = files.upload()

Saving cardio_train.csv to cardio_train.csv


In [5]:
import pandas as pd
df = pd.read_csv('cardio_train.csv', sep=';')

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (70000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [6]:
# Drop ID column
if 'id' in df.columns:
    df.drop('id', axis=1, inplace=True)

# Convert age from days to years
df['age'] = df['age'] / 365

# Check missing values
print(df.isnull().sum())

age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64


In [7]:
# Remove unrealistic blood pressure values
df = df[(df['ap_hi'] < 250) & (df['ap_lo'] < 150)]

# Remove extreme height/weight
df = df[(df['height'] > 120) & (df['height'] < 220)]
df = df[(df['weight'] > 30) & (df['weight'] < 200)]

In [10]:
df.dropna(inplace=True)
df = df.drop_duplicates()
# Remove missing or duplicate entries

# Separate features and labels
X = df.drop('cardio', axis=1)
y = df['cardio']

# Normalize data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split into train/test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Train XGBoost
xgb = XGBClassifier(eval_metric='logloss') # Removed use_label_encoder
xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , LSTM , Conv1D , Flatten

# CNN model
cnn = Sequential([
    Conv1D(32, 2, activation='relu', input_shape=(X_train.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train.reshape(-1, X_train.shape[1], 1), y_train, epochs=5, batch_size=32, verbose=1)

# LSTM model
lstm = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], 1)),
    Dense(1, activation='sigmoid')
])
lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm.fit(X_train.reshape(-1, X_train.shape[1], 1), y_train, epochs=5, batch_size=32, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.7277 - loss: 0.5528
Epoch 2/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7313 - loss: 0.5461
Epoch 3/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7337 - loss: 0.5439
Epoch 4/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7341 - loss: 0.5422
Epoch 5/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7341 - loss: 0.5417
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1722/1722 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.7027 - loss: 0.5802
Epoch 2/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7202 - loss: 0.5622
Epoch 3/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7219 - loss: 0.5580
Epoch 4/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.7243 - loss: 0.5560
Epoch 5/5
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.7262 - loss: 0.5545


In [20]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report

ensemble = VotingClassifier(estimators=[
    ('knn', knn),
    ('xgb', xgb)
], voting='soft')
ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.71      0.75      0.73      6940
           1       0.73      0.68      0.70      6832

    accuracy                           0.72     13772
   macro avg       0.72      0.72      0.72     13772
weighted avg       0.72      0.72      0.72     13772



In [21]:
import numpy as np

X_train = np.array(X_train)
X_test = np.array(X_test)
X_train = X_train.reshape(-1, X_train.shape[1], 1)
X_test = X_test.reshape(-1, X_test.shape[1], 1)


In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

cnn = Sequential([
    Conv1D(64, 2, activation='relu', input_shape=(X_train.shape[1], 1)),
    MaxPooling1D(2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train, y_train, epochs=8, batch_size=32, verbose=1)


Epoch 1/8


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1722/1722 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7227 - loss: 0.5591
Epoch 2/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7277 - loss: 0.5509
Epoch 3/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7299 - loss: 0.5486
Epoch 4/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7312 - loss: 0.5471
Epoch 5/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7312 - loss: 0.5460
Epoch 6/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7325 - loss: 0.5450
Epoch 7/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7330 - loss: 0.5444
Epoch 8/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7326 - loss: 0.5436


In [23]:
from tensorflow.keras.layers import LSTM

lstm = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], 1)),
    Dense(1, activation='sigmoid')
])

lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm.fit(X_train, y_train, epochs=8, batch_size=32, verbose=1)


Epoch 1/8


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1722/1722 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.7009 - loss: 0.5807
Epoch 2/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.7202 - loss: 0.5620
Epoch 3/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - accuracy: 0.7228 - loss: 0.5583
Epoch 4/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.7248 - loss: 0.5561
Epoch 5/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.7257 - loss: 0.5543
Epoch 6/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7274 - loss: 0.5529
Epoch 7/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7282 - loss: 0.5516
Epoch 8/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7301 - loss: 0.5499


In [24]:
# Get predictions from each model
# Use the original 2D X_test for KNN and XGBoost
pred_knn = knn.predict(X_test[:,:,0]) # Assuming the original 2D data is in the first dimension after reshaping
pred_xgb = xgb.predict(X_test[:,:,0]) # Assuming the original 2D data is in the first dimension after reshaping
pred_cnn = (cnn.predict(X_test) > 0.5).astype("int32")
pred_lstm = (lstm.predict(X_test) > 0.5).astype("int32")

# Combine all model predictions (majority voting)
import numpy as np
final_pred = np.round((pred_knn + pred_xgb + pred_cnn.flatten() + pred_lstm.flatten()) / 4)

# Evaluate hybrid model
from sklearn.metrics import accuracy_score
print("Hybrid Model Accuracy:", accuracy_score(y_test, final_pred))

431/431 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
431/431 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Hybrid Model Accuracy: 0.7291606157420853


In [25]:
# Evaluate CNN model with 3D data
loss_cnn, accuracy_cnn = cnn.evaluate(X_test, y_test, verbose=0)
print(f"CNN Model Accuracy with 3D data: {accuracy_cnn}")

# Evaluate LSTM model with 3D data
loss_lstm, accuracy_lstm = lstm.evaluate(X_test, y_test, verbose=0)
print(f"LSTM Model Accuracy with 3D data: {accuracy_lstm}")

CNN Model Accuracy with 3D data: 0.7303224205970764
LSTM Model Accuracy with 3D data: 0.7245861291885376


In [26]:
import numpy as np

X_train = np.array(X_train)
X_test = np.array(X_test)
X_train = X_train.reshape(-1, X_train.shape[1], 1)
X_test = X_test.reshape(-1, X_test.shape[1], 1)



In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

cnn = Sequential([
    Conv1D(64, 2, activation='relu', input_shape=(X_train.shape[1], 1)),
    MaxPooling1D(2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_train, y_train, epochs=8, batch_size=32, verbose=1)


Epoch 1/8


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1722/1722 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7224 - loss: 0.5594
Epoch 2/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7278 - loss: 0.5512
Epoch 3/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7321 - loss: 0.5483
Epoch 4/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7306 - loss: 0.5468
Epoch 5/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7317 - loss: 0.5454
Epoch 6/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.7336 - loss: 0.5447
Epoch 7/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7340 - loss: 0.5438
Epoch 8/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7331 - loss: 0.5431


In [29]:
from tensorflow.keras.layers import LSTM

lstm = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], 1)),
    Dense(1, activation='sigmoid')
])

lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm.fit(X_train, y_train, epochs=8, batch_size=32, verbose=1)


Epoch 1/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.7016 - loss: 0.5814
Epoch 2/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7176 - loss: 0.5628
Epoch 3/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.7225 - loss: 0.5588
Epoch 4/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7226 - loss: 0.5570
Epoch 5/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.7268 - loss: 0.5543
Epoch 6/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7282 - loss: 0.5528
Epoch 7/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7286 - loss: 0.5516
Epoch 8/8
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.7298 - loss: 0.5496


In [30]:
pred_knn = knn.predict(X_test[:,:,0]) # Assuming the original 2D data is in the first dimension after reshaping
pred_xgb = xgb.predict(X_test[:,:,0]) # Assuming the original 2D data is in the first dimension after reshaping
pred_cnn = (cnn.predict(X_test) > 0.5).astype("int32")
pred_lstm = (lstm.predict(X_test) > 0.5).astype("int32")

# Combine all model predictions (majority voting)
import numpy as np
final_pred = np.round((pred_knn + pred_xgb + pred_cnn.flatten() + pred_lstm.flatten()) / 4)

# Evaluate hybrid model
from sklearn.metrics import accuracy_score
print("Hybrid Model Accuracy:", accuracy_score(y_test, final_pred))

431/431 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
431/431 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Hybrid Model Accuracy: 0.7294510601219867


In [31]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Define the parameter grid for XGBoost
# This is a basic example, more parameters can be added
xgb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5]
}

# Create GridSearchCV object
grid_search_xgb = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss'), xgb_params, cv=5, scoring='accuracy')

# Fit the grid search to the data
# Use the 2D training data for XGBoost
grid_search_xgb.fit(X_train[:,:,0], y_train)

# Print the best parameters and best score
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)
print("Best cross-validation accuracy for XGBoost:", grid_search_xgb.best_score_)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:26:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:26:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:26:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:26:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

Best parameters for XGBoost: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
Best cross-validation accuracy for XGBoost: 0.734955069438141


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, LSTM
from scikeras.wrappers import KerasClassifier # Corrected import
from sklearn.model_selection import GridSearchCV
import numpy as np

# Define a function to create the CNN model for KerasClassifier
def create_cnn_model(filters=32, kernel_size=2, dense_units=64):
    model = Sequential([
        Conv1D(filters, kernel_size, activation='relu', input_shape=(X_train.shape[1], 1)),
        MaxPooling1D(2),
        Flatten(),
        Dense(dense_units, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Define a function to create the LSTM model for KerasClassifier
def create_lstm_model(lstm_units=64, dense_units=1):
    model = Sequential([
        LSTM(lstm_units, input_shape=(X_train.shape[1], 1)),
        Dense(dense_units, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Create KerasClassifier for CNN
cnn_classifier = KerasClassifier(build_fn=create_cnn_model, epochs=3, batch_size=32, verbose=0)

# Define parameter grid for CNN
cnn_params = {
    'filters': [32, 64],
    'kernel_size': [2, 3],
    'dense_units': [32, 64]
}

# Create GridSearchCV for CNN
grid_search_cnn = GridSearchCV(cnn_classifier, cnn_params, cv=3, scoring='accuracy')

# Fit GridSearchCV for CNN (using a subset of data for faster tuning)
# Note: For a more thorough tuning, use the full dataset and potentially more epochs/batch size
subset_size = 5000 # Adjust subset size as needed
grid_search_cnn.fit(X_train[:subset_size], y_train[:subset_size])

# Print best parameters and score for CNN
print("Best parameters for CNN:", grid_search_cnn.best_params_)
print("Best cross-validation accuracy for CNN:", grid_search_cnn.best_score_)

# Create KerasClassifier for LSTM
lstm_classifier = KerasClassifier(build_fn=create_lstm_model, epochs=3, batch_size=32, verbose=0)

# Define parameter grid for LSTM
lstm_params = {
    'lstm_units': [32, 64],
    'dense_units': [1] # Usually 1 for binary classification
}

# Create GridSearchCV for LSTM
grid_search_lstm = GridSearchCV(lstm_classifier, lstm_params, cv=3, scoring='accuracy')

# Fit GridSearchCV for LSTM (using the same subset of data)
grid_search_lstm.fit(X_train[:subset_size], y_train[:subset_size])

# Print best parameters and score for LSTM
print("Best parameters for LSTM:", grid_search_lstm.best_params_)
print("Best cross-validation accuracy for LSTM:", grid_search_lstm.best_score_)

In [ ]:
!pip install scikeras

In [32]:
import pickle
pickle.dump(ensemble, open('heart_model.pkl', 'wb'))



In [40]:
from google.colab import files
files.download('heart_model.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
y_pred = ensemble.predict(X_test[:,:,0])

In [37]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7166715074063317

Confusion Matrix:
 [[5212 1728]
 [2174 4658]]

Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.75      0.73      6940
           1       0.73      0.68      0.70      6832

    accuracy                           0.72     13772
   macro avg       0.72      0.72      0.72     13772
weighted avg       0.72      0.72      0.72     13772



In [39]:
pickle.dump(ensemble, open('heart_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

print("✅ Model and Scaler saved successfully!")

✅ Model and Scaler saved successfully!


In [41]:
# Drop ID BEFORE scaling
if 'id' in df.columns:
    df = df.drop('id', axis=1)

X = df.drop('cardio', axis=1)
y = df['cardio']

In [44]:
scaler.fit(X_train_2d)   # Use X_train_2d for scaler
# The line below will cause a NameError as 'model' and 'X_train_scaled' are not defined in this context.
# The scaler and ensemble model were already fitted/trained and saved in previous steps.
# model.fit(X_train_scaled, y_train)

StandardScaler()

In [43]:
import numpy as np

# Reshape X_train and X_test back to 2D (samples, features)
# This assumes the last dimension is 1, as was done for CNN/LSTM
X_train_2d = X_train.reshape(X_train.shape[0], X_train.shape[1])
X_test_2d = X_test.reshape(X_test.shape[0], X_test.shape[1])

print("X_train_2d shape:", X_train_2d.shape)
print("X_test_2d shape:", X_test_2d.shape)

X_train_2d shape: (55085, 11)
X_test_2d shape: (13772, 11)


In [46]:
pickle.dump(ensemble, open('heart_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

In [47]:
print(X_train.shape)

(55085, 11, 1)


In [49]:
import pickle

pickle.dump(ensemble, open('heart_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

In [50]:
from google.colab import files

files.download('heart_model.pkl')
files.download('scaler.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
print(y.value_counts())

cardio
0    34785
1    34072
Name: count, dtype: int64


In [53]:
print(ensemble.predict(X_test_2d[:10]))

[0 0 0 1 1 0 0 0 0 1]


In [55]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, ensemble.predict(X_test_2d)))

0.7166715074063317


In [57]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None]
}

grid = GridSearchCV(RandomForestClassifier(), params, cv=3)
grid.fit(X_train_2d, y_train)

model = grid.best_estimator_

In [60]:
print(ensemble.predict(X_test_2d[:20]))

[0 0 0 1 1 0 0 0 0 1 0 0 0 0 1 0 0 1 1 1]


In [61]:
print(X.columns)

Index(['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol',
       'gluc', 'smoke', 'alco', 'active'],
      dtype='object')


In [62]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Load data
df = pd.read_csv('cardio_train.csv', sep=';')

# DROP ID (VERY IMPORTANT)
df = df.drop('id', axis=1)

# DO NOT CHANGE AGE (keep as it is from dataset)

# Features & target
X = df.drop('cardio', axis=1)
y = df['cardio']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model (KNN — stable)
model = KNeighborsClassifier(n_neighbors=7)
model.fit(X_train_scaled, y_train)

# Check
print(model.predict(X_test_scaled[:10]))  # MUST show mix

# Save
pickle.dump(model, open('heart_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

[1 1 0 1 1 1 0 1 1 1]


In [63]:
pickle.dump(model, open('heart_model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))

In [64]:
from google.colab import files

files.download('heart_model.pkl')
files.download('scaler.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>